# Running the Project on Google Colab

## Standard workflow
1. **Clone code** from GitHub/GitLab into Colab
2. **Mount Google Drive** to read the CICIDS2017 dataset
3. Run the training pipeline

## Prepare data on Google Drive
Upload the 8 CICIDS2017 CSV files to Drive, for example:
```
MyDrive/ids-2017/
```
or
```
MyDrive/data/cicids2017/
```

> Code is cloned to `/content/`; **data is read from Drive**. Colab does not require Java/Spark.

## Step 1: Clone code from GitHub/GitLab

In [ ]:
# GitHub
!git clone https://github.com/vuthainguyen1602/IEEECS_CPS_2026_paper.git

# GitLab: uncomment and update the URL as needed
# !git clone https://gitlab.com/username/IEEECS_CPS_2026_paper.git

## Step 2: Mount Google Drive (read CSV data)

The CICIDS2017 dataset lives on Drive and is **not** included in the Git repo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3A: Find CSV files on Drive (run if `CSV files: 0`)

This cell lists folders containing `.csv` files on Drive so you can set `RAW_DATA_DIR` correctly.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
search_roots = [DRIVE_ROOT]

print("Searching for .csv files on Google Drive...\n")
found_dirs = {}

for root in search_roots:
    if not root.exists():
        print(f"Path not found: {root}. Run the Drive mount step first.")
        continue

    for csv_path in root.rglob("*.csv"):
        folder = str(csv_path.parent)
        found_dirs.setdefault(folder, 0)
        found_dirs[folder] += 1

    for csv_path in root.rglob("*.CSV"):
        folder = str(csv_path.parent)
        found_dirs.setdefault(folder, 0)
        found_dirs[folder] += 1

if not found_dirs:
    print("No CSV files found on Drive.")
    print("\nCheck:")
    print("1. Have you uploaded all 8 CICIDS2017 CSV files?")
    print("2. Are the files still inside a .zip archive?")
    print("3. Downloads are often in the MachineLearningCVE/ folder")
    print("\nDrive root contents:")
    !ls -la /content/drive/MyDrive
else:
    print("Folders containing CSV files:")
    for folder, count in sorted(found_dirs.items(), key=lambda x: -x[1]):
        print(f"  [{count} files] {folder}")
    print("\n=> Copy one of the paths above into RAW_DATA_DIR in Step 3B")

## Step 3B: Configure paths

Copy a path from Step 3A into `RAW_DATA_DIR`.

Common examples:
- `/content/drive/MyDrive/ids-2017`
- `/content/drive/MyDrive/ids-2017/MachineLearningCVE`

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = "/content/IEEECS_CPS_2026_paper/paper_1"

# UPDATE this path based on the result from Step 3A
RAW_DATA_DIR = "/content/drive/MyDrive/ids-2017"

def count_csv_files(folder: Path) -> int:
    return len(list(folder.glob("*.csv"))) + len(list(folder.glob("*.CSV")))

if not Path(PROJECT_DIR).exists():
    raise FileNotFoundError(
        f"Path not found: {PROJECT_DIR}. Run Step 1 (git clone) first."
    )

raw_dir = Path(RAW_DATA_DIR)
if not raw_dir.exists():
    raise FileNotFoundError(
        f"Path not found: {RAW_DATA_DIR}. Run Step 3A to locate the correct CSV folder."
    )

os.chdir(PROJECT_DIR)
csv_count = count_csv_files(raw_dir)

print("Working directory:", os.getcwd())
print("Raw CSV directory (Drive):", RAW_DATA_DIR)
print("CSV files found:", csv_count)
print("\nDirectory contents:")
!ls -la "{RAW_DATA_DIR}"

if csv_count == 0:
    raise FileNotFoundError(
        f"No .csv files found in {RAW_DATA_DIR}.\n"
        "Try appending /MachineLearningCVE or rerun Step 3A."
    )

## Step 4: Install dependencies

In [ ]:
!pip install -r requirements.txt

## Step 5: Prepare data from Google Drive

Read CSV files from Drive and save Parquet files to `paper_1/data/` (in Colab).

In [ ]:
!python code/data_preparation.py --input-dir {RAW_DATA_DIR}

## Step 6: Train models

In [ ]:
!python code/hybrid_ids_cicids2017.py
!python code/knowledge_distillation.py

## Step 7: Update figures and LaTeX tables

In [ ]:
!python code/replot_figures.py
!python code/populate_latex_tables.py